In [0]:
%sql

-- =========================================================
-- DDL Bronze: pf.bronze.models_raw
-- Origen: CSV del Volumen landing (spark.read.csv)
-- Estrategia: append, particionado por ingestion_date
-- =========================================================

DROP TABLE IF EXISTS pf.bronze.models_raw;

CREATE TABLE IF NOT EXISTS pf.bronze.models_raw (
    _id            STRING   COMMENT 'ID interno de Hugging Face',
    id             STRING   COMMENT 'ID natural org/nombre (business key)',
    modelId        STRING   COMMENT 'Alias del ID',
    likes          STRING   COMMENT 'Likes (snapshot del dia)',
    private        STRING  COMMENT 'Repo privado',
    downloads      STRING   COMMENT 'Descargas acumuladas (snapshot del dia)',
    pipeline_tag   STRING   COMMENT 'Tarea del modelo',
    library_name   STRING   COMMENT 'Libreria (transformers, vLLM, ...)',
    createdAt      STRING   COMMENT 'Fecha creacion (ISO, sin normalizar)',
    lastModified   STRING   COMMENT 'Ultima modificacion (ISO)',
    tags           STRING   COMMENT 'JSON array de tags',
    payload_json   STRING   COMMENT 'Backup integro del JSON original de la API',
    ingesta_run_id STRING   COMMENT 'Corrida que genero el registro',
    ingesta_mode   STRING   COMMENT 'historical | incremental | full_refresh',
    page_no        STRING   COMMENT 'Pagina de procedencia dentro de la corrida',
    ingestion_ts   TIMESTAMP COMMENT 'Timestamp de ingesta (UTC)',
    ingestion_date DATE     COMMENT 'Particion: fecha util de ingesta',
    _rescued_data  STRING   COMMENT 'Campos no mapeados (schema evolution)'
)
USING DELTA
PARTITIONED BY (ingestion_date)
TBLPROPERTIES (
   delta.enableChangeDataFeed      = true,
   delta.autoOptimize.optimizeWrite = true,
   delta.autoOptimize.autoCompact  = true,
   delta.feature.allowColumnDefaults = 'supported'
)
COMMENT 'Bronze: modelos crudos leidos desde el CSV del Volumen landing';

-- Verificación
SELECT * FROM pf.bronze.models_raw LIMIT 5;
SELECT COUNT(*) AS n_rows FROM pf.bronze.models_raw;